# Example 3: Agentic Workflow Executor for Multi-Step Business Tasks
Companion notebook for **Example 3** of Chapter 6.

One node breaks a request into a plan, one selects the next step, one executes it through MCP, and one verifies the result before moving on -- controlled autonomy: each step is visible, validated, and recoverable. Self-contained -- run this notebook on its own.

In [1]:
from typing import TypedDict


class WorkflowState(TypedDict, total=False):
    user_request: str
    plan: list[dict]
    current_step: dict
    completed_steps: list[dict]
    artifacts: dict
    last_result: dict
    last_error: str
    next_action: str
    status: str

`plan` holds structured steps like `{"tool": "create_vendor", "args": {...}}`. `artifacts` stores outputs later steps need, such as a new `vendor_id`. Keeping these in shared state (not just chat history) lets the graph continue after retries or restarts.

In [2]:
def decompose_task(state: WorkflowState) -> WorkflowState:
    return {
        "plan": [
            {"tool": "create_vendor", "args": {"name": "Acme Parts"}},
            {"tool": "send_message", "args": {"team": "finance", "template": "vendor_created"}},
            {"tool": "update_tracker", "args": {"stage": "created"}},
        ],
        "completed_steps": [],
        "artifacts": {},
    }

In [3]:
def select_next_step(state: WorkflowState) -> WorkflowState:
    """Pop the next unexecuted step off the plan, or signal the plan is done."""
    plan = state["plan"]
    completed = state["completed_steps"]
    if len(completed) >= len(plan):
        return {"next_action": "done"}
    return {"current_step": plan[len(completed)], "next_action": "execute"}

## A small MCP server for the business tools
`create_vendor`, `send_message`, `update_tracker` -- the same business capabilities behind a standard MCP interface, whether they'd really be CRM updates, messaging, or internal system writes.

In [4]:
%%writefile workflow_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("WorkflowServer")

_vendor_counter = 1000


@mcp.tool()
def create_vendor(name: str) -> dict:
    """Create a vendor record and return its details."""
    global _vendor_counter
    if not name.strip():
        raise ValueError("vendor name is required")
    _vendor_counter += 1
    return {"ok": True, "vendor_id": f"V-{_vendor_counter}", "name": name, "status": "created"}


@mcp.tool()
def send_message(team: str, template: str) -> dict:
    """Send a workflow notification message to an internal team."""
    if not team.strip() or not template.strip():
        raise ValueError("team and template are required")
    return {"ok": True, "team": team, "template": template, "delivery_status": "sent"}


@mcp.tool()
def update_tracker(stage: str) -> dict:
    """Create or update the onboarding tracker stage."""
    if not stage.strip():
        raise ValueError("stage is required")
    return {"ok": True, "stage": stage, "tracker_status": "updated"}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing workflow_server.py


**Execute node** -- one step at a time. A tool failure is caught and recorded rather than crashing the graph, so the workflow can decide what to do next:

In [5]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, START, END

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [6]:
async def execute_step(state: WorkflowState, mcp: ClientSession) -> WorkflowState:
    step = state["current_step"]
    try:
        result = await call_tool(mcp, step["tool"], step["args"])
        return {"last_result": result, "last_error": ""}
    except Exception as e:
        return {"last_error": str(e), "status": "needs_replan"}

**Verify node** -- checks the output *before* moving ahead (e.g. did `create_vendor` return a real ID?). If verification fails, the workflow stops for review instead of silently continuing; a production system could instead retry or replan here with a safer action.

In [7]:
def verify_step(state: WorkflowState) -> WorkflowState:
    step = state["current_step"]
    result = state.get("last_result", {})
    tool = step["tool"]

    valid = {
        "create_vendor": bool(result.get("vendor_id")),
        "send_message": result.get("delivery_status") == "sent",
        "update_tracker": result.get("tracker_status") == "updated",
    }.get(tool, False)

    if not valid:
        return {"status": "needs_review", "last_error": f"verification failed for {tool}"}

    completed = state["completed_steps"] + [{"tool": tool, "result": result}]
    artifacts = dict(state["artifacts"])
    if tool == "create_vendor":
        artifacts["vendor_id"] = result["vendor_id"]
    return {"completed_steps": completed, "artifacts": artifacts, "status": "verified"}

In [8]:
def route_after_select(state: WorkflowState) -> str:
    return "finish" if state["next_action"] == "done" else "execute"


def route_after_execute(state: WorkflowState) -> str:
    return "finish" if state.get("status") == "needs_replan" else "verify"


def route_after_verify(state: WorkflowState) -> str:
    return "select" if state.get("status") == "verified" else "finish"

## Wiring the graph
`decompose -> select -> execute -> verify -> (loop back to select, or stop)`:

In [9]:
def build_graph(mcp_session: ClientSession):
    graph = StateGraph(WorkflowState)

    async def execute_node(state):
        return await execute_step(state, mcp_session)

    graph.add_node("decompose", decompose_task)
    graph.add_node("select", select_next_step)
    graph.add_node("execute", execute_node)
    graph.add_node("verify", verify_step)

    graph.add_edge(START, "decompose")
    graph.add_edge("decompose", "select")
    graph.add_conditional_edges("select", route_after_select, {"execute": "execute", "finish": END})
    graph.add_conditional_edges("execute", route_after_execute, {"verify": "verify", "finish": END})
    graph.add_conditional_edges("verify", route_after_verify, {"select": "select", "finish": END})
    return graph.compile()

In [10]:
server_path = Path.cwd() / "workflow_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        app = build_graph(session)
        result = await app.ainvoke({
            "user_request": "Create a vendor record, notify finance, and update the onboarding tracker."
        })

print("status:", result.get("status"))
print("completed steps:", len(result["completed_steps"]))
for step in result["completed_steps"]:
    print(f"  - {step['tool']}: {step['result']}")
print("artifacts:", result["artifacts"])
if result.get("last_error"):
    print("last_error:", result["last_error"])

status: verified
completed steps: 3
  - create_vendor: {'ok': True, 'vendor_id': 'V-1001', 'name': 'Acme Parts', 'status': 'created'}
  - send_message: {'ok': True, 'team': 'finance', 'template': 'vendor_created', 'delivery_status': 'sent'}
  - update_tracker: {'ok': True, 'stage': 'created', 'tracker_status': 'updated'}
artifacts: {'vendor_id': 'V-1001'}
